# Simple Local RAG System

**RAG = Retrieval-Augmented Generation**

Instead of relying only on what the AI memorized during training, RAG lets the AI **look up** information from *your own documents* before answering.

```
Your Question
     │
     ▼
[ Vector Search ] ──▶  Find the most relevant docs in your folder
     │
     ▼
[ LLM + Context ] ──▶  Generate an answer grounded in YOUR documents
```

### What we will build
| Step | What happens |
|------|--------------|
| 1 | Install libraries |
| 2 | Convert documents into vectors and store them in ChromaDB |
| 3 | Retrieve the most relevant documents for a question |
| 4 | Generate an answer using Groq API + retrieved context |

> Place your own `.txt` files in the `knowledge_base/` folder before running Step 2.

### Libraries used
- **chromadb** – local vector database (stores and searches embeddings)
- **sentence-transformers** – converts text into vectors (embeddings)
- **groq** – Groq API for the generation step

---
## Step 1: Install Libraries

In [ ]:
!pip install chromadb sentence-transformers groq python-dotenv

---
## Step 2: Load, Chunk, and Build the Vector Database

We split this into three sub-steps:
- **Load** – read all `.txt` files from the folder
- **Chunk** – split each document into smaller overlapping pieces
- **Embed & Store** – convert chunks to vectors and save them in ChromaDB

### Why chunk?
Embedding a whole document as one vector loses detail — the vector becomes a blurry average of everything in the file.
By splitting into smaller chunks (e.g. 300 characters with 50-character overlap) we can retrieve only the *specific passage* that answers the question.

```
Document (500 words)
    │
    ▼
chunk_0  chunk_1  chunk_2  chunk_3   ← each gets its own vector
   ↕        ↕        ↕        ↕
  overlap  overlap  overlap
```

> The embedding model `all-MiniLM-L6-v2` (~80 MB) will download automatically on the first run.

In [8]:
from pathlib import Path

DOCS_FOLDER = "./knowledge_base"  # ← change this to point at a different folder

# ── Load documents ────────────────────────────────────────────────────────────
def load_documents(folder_path: str) -> list[dict]:
    documents = []
    for filepath in Path(folder_path).glob("*.txt"):
        text = filepath.read_text(encoding="utf-8")
        documents.append({"id": filepath.stem, "text": text, "source": filepath.name})
    return documents

# ── Chunk text ────────────────────────────────────────────────────────────────
def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    """Split text into overlapping fixed-size chunks."""
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end].strip())
        if end == len(text):
            break
        start += chunk_size - overlap
    return [c for c in chunks if c]

# ── Apply chunking to all documents ──────────────────────────────────────────
documents = load_documents(DOCS_FOLDER)

all_chunks = []
for doc in documents:
    for i, chunk in enumerate(chunk_text(doc["text"])):
        all_chunks.append({
            "id":     f"{doc['id']}_chunk_{i}",
            "text":   chunk,
            "source": doc["source"],
        })

print(f"Loaded {len(documents)} documents → split into {len(all_chunks)} chunks")
print(f"\nExample: first 2 chunks from '{documents[0]['source']}':")
for chunk in all_chunks[:2]:
    print(f"\n  [{chunk['id']}]\n  {chunk['text']}")

Loaded 4 documents → split into 14 chunks

Example: first 2 chunks from 'machine_learning.txt':

  [machine_learning_chunk_0]
  Machine learning (ML) is a subset of artificial intelligence that allows computers
to learn from data without being explicitly programmed for every task.

Types of machine learning:
1. Supervised Learning  – the model learns from labeled examples (e.g. spam detection)
2. Unsupervised Learning – the

  [machine_learning_chunk_1]
  g. spam detection)
2. Unsupervised Learning – the model finds patterns in unlabeled data (e.g. clustering)
3. Reinforcement Learning – the model learns by trial and error using rewards

Key concepts:
- Training data: labeled examples used to teach the model
- Features: input variables the model uses


In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

# ── Load the embedding model ──────────────────────────────────────────────────
print("Loading embedding model (downloads ~80 MB on first run)...")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model ready!")

# ── Embed all chunks ──────────────────────────────────────────────────────────
texts = [c["text"]   for c in all_chunks]
ids   = [c["id"]     for c in all_chunks]
metas = [{"source": c["source"]} for c in all_chunks]

print("\nCreating embeddings...")
embeddings = embedding_model.encode(texts, show_progress_bar=True)
print(f"Each chunk is now a vector of {len(embeddings[0])} numbers")

# ── Store in ChromaDB ─────────────────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(path="./chroma_db")
try:
    chroma_client.delete_collection("knowledge_base")
except Exception:
    pass

collection = chroma_client.create_collection(
    name="knowledge_base",
    metadata={"hnsw:space": "cosine"},
)

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metas,
)

print(f"\nStored {collection.count()} chunks in ChromaDB (saved to ./chroma_db/)")

C:\Users\Tsvetomir.Teach\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model (downloads ~80 MB on first run)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3215.65it/s]


Model ready!

Creating embeddings...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.61it/s]


Each chunk is now a vector of 384 numbers

Stored 21 chunks in ChromaDB (saved to ./chroma_db/)


---
## Step 3: Retrieve Relevant Documents

Given a question, we:
1. Embed the question into a vector
2. Find the documents whose vectors are **closest** (most similar) to the question vector

This is the **Retrieval** part of RAG.

In [4]:
def retrieve(query: str, n_results: int = 2) -> list[dict]:
    """Find the most relevant documents for a given query."""
    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )

    retrieved = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        retrieved.append({
            "text":       doc,
            "source":     meta["source"],
            "similarity": round(1 - dist, 3),  # convert distance → similarity score
        })

    return retrieved


# ── Test retrieval ────────────────────────────────────────────────────────────
# test_query = "What is machine learning?"
test_query = "Who is the president of France?"
results = retrieve(test_query, n_results=2)

print(f'Query: "{test_query}"\n')
for i, result in enumerate(results, 1):
    print(f"{i}. Source: {result['source']}  |  Similarity: {result['similarity']}")
    print(f"   Preview: {result['text'][:120].strip()}...")
    print()

Query: "Who is the president of France?"

1. Source: emmanuel_macron.txt  |  Similarity: 0.633
   Preview: ed as President of France.
- He was re-elected for a second five-year term in April 2022, defeating Marine Le Pen in the...

2. Source: emmanuel_macron.txt  |  Similarity: 0.616
   Preview: Emmanuel Macron – President of France

Emmanuel Jean-Michel Frédéric Macron was born on December 21, 1977, in Amiens, Fr...



---
## Step 4: Generate an Answer (RAG = Retrieval + Generation)

Now we combine retrieval with a language model:
1. Retrieve relevant documents
2. Pass them as **context** to Llama 3.1 8B (running on Groq)
3. The model answers using only the provided context

> **Before running this cell:** create a `.env` file in the same folder as this notebook with your Groq API key (free at [console.groq.com](https://console.groq.com)):
> ```
> GROQ_API_KEY=gsk_...
> ```
> The `.env` file is the recommended way to handle API keys in Jupyter on Windows — environment variables set via System Properties are often not visible to the kernel.

In [6]:
import os
import httpx
import warnings
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

# Corporate/school networks often use SSL inspection proxies with self-signed
# certificates, which causes SSL verification errors. This disables that check.
warnings.filterwarnings("ignore", message="Unverified HTTPS request")
groq_client = Groq(http_client=httpx.Client(verify=False))


def rag_answer(question: str, n_results: int = 2) -> str:
    """
    Full RAG pipeline:
      1. Retrieve the most relevant documents for the question
      2. Build a prompt with those documents as context
      3. Ask Llama 3.1 8B to answer using only the provided context
    """
    # Step 1 – Retrieve
    relevant_docs = retrieve(question, n_results=n_results)

    # Step 2 – Build context string
    context = "\n\n---\n\n".join(
        f"[Source: {doc['source']}]\n{doc['text']}"
        for doc in relevant_docs
    )

    # Step 3 – Generate
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        max_tokens=512,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant. "
                    "Answer the user's question using ONLY the context provided below. "
                    "If the context does not contain enough information, say so clearly."
                ),
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion: {question}",
            },
        ],
    )

    return response.choices[0].message.content


# ── Test the full RAG pipeline ────────────────────────────────────────────────
question = "What is machine learning and how does it relate to neural networks?"
print(f"Q: {question}\n")
answer = rag_answer(question)
print(f"A: {answer}")

Q: What is machine learning and how does it relate to neural networks?

A: I don't have enough information to answer your question. The provided context only contains information about Emmanuel Macron, the President of France. It does not contain any information about machine learning or neural networks.


---
## Step 5: Play around with our own Questions!

In [7]:
my_questions = [
    "What is RAG and why should I use it?",
    "How does backpropagation work?",
    "What are some tips for writing better Python code?",
    # Add your own questions here!
]

for q in my_questions:
    print(f"Q: {q}")
    print(f"A: {rag_answer(q)}")
    print("-" * 70)

Q: What is RAG and why should I use it?
A: I couldn't find any information about RAG in the provided context.
----------------------------------------------------------------------
Q: How does backpropagation work?
A: I don't have information about backpropagation in the provided context.
----------------------------------------------------------------------
Q: What are some tips for writing better Python code?
A: I don't have any information on Python or any advice for writing Python code, as both contexts provided are about unrelated topics (climate policy, a government official, and another context about a government leader).
----------------------------------------------------------------------


---
## Summary

You just built a working RAG system with only ~60 lines of code!

```
knowledge_base/          ← your documents live here
    machine_learning.txt
    neural_networks.txt
    rag_explained.txt
    python_tips.txt

chroma_db/               ← ChromaDB stores the chunk vectors here (persists on disk)
```

### Key concepts to remember

| Concept | What it does |
|---------|-------------|
| **Chunking** | Splits documents into small overlapping pieces for precise retrieval |
| **Embedding** | Converts a chunk into a vector (list of numbers) that captures meaning |
| **Vector DB** | Stores vectors and finds the closest ones to a query vector |
| **Cosine similarity** | Measures how "close" two vectors are (1 = identical, 0 = unrelated) |
| **Context window** | The retrieved chunks passed to the LLM as extra information |

### Tunable parameters

| Parameter | Default | Effect |
|-----------|---------|--------|
| `chunk_size` | 300 chars | Smaller = more precise retrieval; larger = more context per chunk |
| `overlap` | 50 chars | Prevents losing information at chunk boundaries |
| `n_results` | 2 | How many chunks to retrieve per question |

### What to try next
- **Change `chunk_size` and `overlap`** – see how it affects the retrieved passages
- **Point it at your own documents** – change `DOCS_FOLDER` to any folder
- **Add more file types** – load `.pdf` or `.docx` with `pypdf` or `python-docx`
- **Try semantic chunking** – split on paragraph boundaries instead of fixed character counts